# 08 · Stay lazy until the answer is requested

## Context

Real cube archives quickly outgrow memory. Lazy arrays let us describe a
method first and materialize only the final analysis.

## Question

Can a composed pipe preserve Dask-backed execution while computing a compact
spatial summary from real observations?

## Analysis story

We open the reviewed fixture with chunks, compose anomaly and variance verbs,
confirm that the result stays lazy, and compute only the final map.


### Data used in this lesson

Every value comes from the PRISM Group at Oregon State University's AN91d
daily 4 km climate product. This repository carries a small Boulder-region
extract for 1–30 January 2024 so the lesson runs offline without replacing
observations with generated values. The [data validation page](../validation/data.md)
records source URLs, terms, checksums, bounds, units, and acceptance tests.

## Prepare · Open the official extract as a chunked cube

In [ ]:
from pathlib import Path
import xarray as xr

# The checked-in extract makes the lesson reproducible without a live service.
data_path = next(
    candidate / "tests" / "fixtures" / "real_data" / "prism_boulder_january_2024.nc"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "tests" / "fixtures" / "real_data" / "prism_boulder_january_2024.nc").exists()
)
cube = xr.open_dataset(
    data_path,
    engine="scipy",
    chunks={"time": 10, "y": 12, "x": 12},
)["tmax"]

assert cube.attrs["is_synthetic"] == 0
assert hasattr(cube.data, "chunks")
cube

## Pipe · Describe the method without triggering computation

In [ ]:
from cubedynamics import pipe, verbs as v

lazy_variability = (
    pipe(cube)
    | v.anomaly(dim="time")
    | v.variance(dim="time")
).unwrap()

assert hasattr(lazy_variability.data, "chunks")
lazy_variability

## Figure · Compute only the final spatial answer

In [ ]:
import matplotlib.pyplot as plt

variability = lazy_variability.compute()
assert not hasattr(variability.data, "chunks")

fig, ax = plt.subplots(figsize=(6.5, 4.5), constrained_layout=True)
variability.plot(ax=ax, cmap="viridis", cbar_kwargs={"label": "Anomaly variance (°C²)"})
ax.set_title("January maximum-temperature variability")
plt.show()

## What the figure tells us

The final map identifies locations with more variable departures while the
intermediate anomaly cube remained lazy.

## Try the next variation

Change chunk sizes and confirm that final values remain identical.